# EcoVLM -- InternVL3.5-8B DAP (ABTP/DAP) -- Full Run Notebook
Clean, commit-ready notebook. All debugging/diagnostic cells from development removed. Produces `internvl3.5-8b-dap_telemetry.csv` with paired `unpruned_matched` / `dap` rows per sample, across 5 runs.

## Setup

In [ ]:
!pip install -U bitsandbytes "transformers==4.57.1" accelerate timm einops zeus-ml -q

**Restart session after the above cell**, then continue from here.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", os.environ["HF_TOKEN"][:8], "...")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Visible devices:", torch.cuda.device_count())
print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import time
from zeus.monitor import ZeusMonitor
monitor = ZeusMonitor(gpu_indices=[0])
print("Idle power (W):", monitor.gpus.getInstantPowerUsage(0) / 1000)

## Model load
No global `attn_implementation="eager"` -- the model defaults to eager on this checkpoint, so we explicitly force SDPA after load (checkpoint layers bypass this via a direct `eager_attention_forward` call inside their patched forward, so they still get real attention weights).

In [ ]:
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = "OpenGVLab/InternVL3_5-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map={"": 0},
).eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
layers = model.language_model.model.layers

# Force SDPA globally for memory reasons -- checkpoint layers bypass this via direct
# eager_attention_forward calls, so they still produce real attention weights.
model.language_model.config._attn_implementation = "sdpa"

print("Loaded on:", next(model.parameters()).device)
print("attn_implementation:", model.language_model.config._attn_implementation)
print("Number of decoder layers:", len(layers))
print("GPU memory (GB):", torch.cuda.memory_allocated(0) / 1e9)

## Image preprocessing helpers (InternVL dynamic tiling)

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))

    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_path, input_size=448, max_num=6):
    image = Image.open(image_path).convert("RGB")
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(img) for img in images]
    return torch.stack(pixel_values)

## DAP core: mean-based importance scoring

In [ ]:
def compute_mean_importance(attn_weights, visual_indices):
    """
    attn_weights: [1, num_heads, seq, seq] from one eager-computed layer
    visual_indices: 1D tensor of candidate (visual) token positions in CURRENT
    (possibly already-pruned) coordinate space.
    Returns: importance score per visual token -- mean attention RECEIVED,
    averaged over heads and over causally-valid query positions (i >= j).
    """
    device = attn_weights.device
    seq_len = attn_weights.shape[-1]
    aw = attn_weights[0].mean(dim=0)  # average over heads -> [seq, seq] (query, key)

    seq_range = torch.arange(seq_len, device=device)
    valid_counts = seq_len - seq_range  # key j: valid queries i in [j, seq_len-1] -> count = seq_len - j

    col_sums = aw.sum(dim=0)  # sum over query dim for each key -> [seq]
    importance_all = col_sums / valid_counts.clamp(min=1).to(col_sums.dtype)

    return importance_all[visual_indices.to(device)]

## DAP core: full generation with 3-checkpoint deferral pruning
Calling with `checkpoint_layers=[]` disables pruning entirely (used for the matched unpruned control) while reusing the exact same code path, cache mechanics, and decode loop -- this keeps the unpruned-vs-DAP comparison as close to apples-to-apples as possible.

In [ ]:
import types
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask
from transformers.models.qwen3.modeling_qwen3 import apply_rotary_pos_emb, eager_attention_forward

CHECKPOINT_LAYERS = [10, 14, 18]   # rescaled per design brief (36-layer Qwen3-8B backbone)
KEEP_PCTILE = 70                  # >= this percentile of current candidates: keep outright
DROP_PCTILE = 30                  # <= this percentile: drop outright; middle band defers

def dap_generate(model, tokenizer, pixel_values, question, max_new_tokens=32,
                  checkpoint_layers=CHECKPOINT_LAYERS, keep_pctile=KEEP_PCTILE, drop_pctile=DROP_PCTILE):
    device = next(model.parameters()).device
    lm = model.language_model.model
    layers_ = lm.layers
    config = lm.config

    # --- Intercept InternVL's own real embedding merge + input_ids, via a deliberate early stop ---
    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        raise RuntimeError("OK1")
    orig_lm_gen = model.language_model.generate
    model.language_model.generate = spy_lm_generate
    probe_config = dict(max_new_tokens=1, do_sample=False)
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = orig_lm_gen

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    orig_model_gen = model.generate
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = orig_model_gen

    inputs_embeds = intercepted_local['inputs_embeds']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == model.img_context_token_id).squeeze(0)
    visual_indices = torch.nonzero(vis_mask, as_tuple=True)[0].to(device)

    def build_mask_pos(h, attn_mask_1d, cache_position, past_kv):
        pos_ids = cache_position.unsqueeze(0)
        mask_kwargs = {
            "config": config, "input_embeds": h, "attention_mask": attn_mask_1d,
            "cache_position": cache_position, "past_key_values": past_kv, "position_ids": pos_ids,
        }
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        pos_emb = lm.rotary_emb(h, pos_ids)
        return causal_mask_mapping, pos_emb, pos_ids

    log = {"n_visual_tokens": visual_indices.numel(), "checkpoints": []}

    with torch.no_grad():
        past_key_values = DynamicCache(config=config)
        hidden_states = inputs_embeds
        deferred_local = visual_indices.clone()

        # --- Prefill: manual layer-by-layer loop, pruning at checkpoint layers ---
        for i, decoder_layer in enumerate(layers_[: config.num_hidden_layers]):
            cur_len = hidden_states.shape[1]
            cache_position = torch.arange(0, cur_len, device=device)
            attn_mask_1d = torch.ones(1, cur_len, device=device, dtype=torch.long)
            causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(
                hidden_states, attn_mask_1d, cache_position, past_key_values
            )

            if i in checkpoint_layers:
                is_final = (i == checkpoint_layers[-1])
                attn_module = decoder_layer.self_attn
                original_fwd = attn_module.forward
                local_capture = {}

                def scoring_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
                    input_shape = hidden_states.shape[:-1]
                    hidden_shape = (*input_shape, -1, self.head_dim)
                    q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                    k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                    v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
                    cos, sin = position_embeddings
                    q, k = apply_rotary_pos_emb(q, k, cos, sin)
                    if past_key_values is not None:
                        k, v = past_key_values.update(k, v, self.layer_idx)
                    attn_output, attn_weights = eager_attention_forward(
                        self, q, k, v, attention_mask, dropout=0.0, scaling=self.scaling
                    )
                    local_capture['weights'] = attn_weights.detach()
                    attn_output = attn_output.reshape(*input_shape, -1).contiguous()
                    attn_output = self.o_proj(attn_output)
                    return attn_output, None

                attn_module.forward = types.MethodType(scoring_forward, attn_module)
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
                attn_module.forward = original_fwd

                if deferred_local.numel() > 0:
                    importance = compute_mean_importance(local_capture['weights'], deferred_local)
                    if is_final:
                        median_cut = torch.median(importance.float())
                        keep_mask = importance >= median_cut
                        keep_idx = deferred_local[keep_mask]
                        drop_idx = deferred_local[~keep_mask]
                        new_deferred = torch.tensor([], dtype=torch.long, device=device)
                    else:
                        keep_cut = torch.quantile(importance.float(), keep_pctile / 100.0)
                        drop_cut = torch.quantile(importance.float(), drop_pctile / 100.0)
                        keep_mask = importance >= keep_cut
                        drop_mask = (importance <= drop_cut) & (~keep_mask)
                        defer_mask = (~keep_mask) & (~drop_mask)
                        keep_idx = deferred_local[keep_mask]
                        drop_idx = deferred_local[drop_mask]
                        new_deferred = deferred_local[defer_mask]
                else:
                    keep_idx = torch.tensor([], dtype=torch.long, device=device)
                    drop_idx = torch.tensor([], dtype=torch.long, device=device)
                    new_deferred = deferred_local

                log["checkpoints"].append({
                    "layer": i, "candidates": deferred_local.numel(),
                    "kept": keep_idx.numel(), "dropped": drop_idx.numel(), "deferred": new_deferred.numel(),
                })

                # Remap surviving indices into new (post-prune) coordinate space
                cur_seq_len = hidden_states.shape[1]
                all_idx = torch.arange(cur_seq_len, device=device)
                is_candidate = torch.zeros(cur_seq_len, dtype=torch.bool, device=device)
                is_candidate[deferred_local] = True
                non_candidate_idx = all_idx[~is_candidate]
                surviving = torch.sort(torch.cat([non_candidate_idx, keep_idx, new_deferred])).values
                remap = torch.full((cur_seq_len,), -1, dtype=torch.long, device=device)
                remap[surviving] = torch.arange(surviving.numel(), device=device)

                hidden_states = hidden_states[:, surviving, :]
                deferred_local = remap[new_deferred] if new_deferred.numel() > 0 else new_deferred

                # Prune this layer's just-written KV cache to match survivors
                past_key_values.layers[i].keys = past_key_values.layers[i].keys[:, :, surviving, :]
                past_key_values.layers[i].values = past_key_values.layers[i].values[:, :, surviving, :]
            else:
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )

        hidden_states = lm.norm(hidden_states)
        logits = model.language_model.lm_head(hidden_states[:, -1:, :])
        next_token = logits.argmax(dim=-1)
        generated_ids = [next_token.item()]

        # --- Decode: per-layer cache lengths now differ by checkpoint segment ---
        for step in range(max_new_tokens - 1):
            if next_token.item() == tokenizer.eos_token_id:
                break
            hidden_states = model.language_model.get_input_embeddings()(next_token)
            for i, decoder_layer in enumerate(layers_[: config.num_hidden_layers]):
                seg_cache_len = past_key_values.get_seq_length(layer_idx=i)
                cache_position = torch.arange(seg_cache_len, seg_cache_len + 1, device=device)
                attn_mask_1d = torch.ones(1, seg_cache_len + 1, device=device, dtype=torch.long)
                causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(
                    hidden_states, attn_mask_1d, cache_position, past_key_values
                )
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
            hidden_states = lm.norm(hidden_states)
            logits = model.language_model.lm_head(hidden_states[:, -1:, :])
            next_token = logits.argmax(dim=-1)
            generated_ids.append(next_token.item())

    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response, len(generated_ids), log

## Constants, data, accuracy scorer

In [ ]:
import json, csv, gc
from datetime import datetime, timezone

MODEL_NAME = "internvl3.5-8b-dap"
DATA_DIR = "/kaggle/input/notebooks/mmrafi14/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")
OUT_DIR = "/kaggle/working"
LOG_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_telemetry.csv")
MANIFEST_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_manifest.json")

N_RUNS = 5
MAX_NEW_TOKENS = 32
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."
MAX_TILES = 6

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)

print(f"Samples loaded: {len(SAMPLES)}")
print(f"Log file: {LOG_FILE}")
print(f"N_RUNS: {N_RUNS}")
print(f"CHECKPOINT_LAYERS: {CHECKPOINT_LAYERS}")
assert N_RUNS == 5, f"N_RUNS is {N_RUNS}, expected 5 -- check this cell before committing"

In [ ]:
def vqa_accuracy(predicted: str, ground_truth_answers: list) -> float:
    import re
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for article in ["a ", "an ", "the "]:
            if s.startswith(article):
                s = s[len(article):]
        return s.strip()
    pred_norm = normalise(predicted)
    matches = sum(1 for a in ground_truth_answers if normalise(a) == pred_norm)
    return min(matches / 3, 1.0)

## Manifest (resume support across commit sessions)

In [ ]:
def load_manifest():
    if os.path.exists(MANIFEST_FILE):
        with open(MANIFEST_FILE) as f:
            return json.load(f)
    return {"completed_runs": [], "current_run": None}

def save_manifest(manifest):
    with open(MANIFEST_FILE, "w") as f:
        json.dump(manifest, f, indent=2)

manifest = load_manifest()
print("Completed runs so far:", manifest["completed_runs"])

## Idle baseline

In [ ]:
print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")
print(f"Idle std dev:  {(sum((x-IDLE_POWER_W)**2 for x in idle_readings)/len(idle_readings))**0.5:.3f} W")

## Measurement loop
Two rows per sample per run: `unpruned_matched` (checkpoint_layers=[], same custom generation path, pruning disabled) and `dap` (full 3-checkpoint pruning). This keeps the energy/accuracy comparison on matched footing -- both methods run through the identical custom loop, differing only in whether pruning triggers.

In [ ]:
write_header = not (os.path.exists(LOG_FILE) and os.path.getsize(LOG_FILE) > 0)
log_f = open(LOG_FILE, "a", newline="", buffering=1)
fieldnames = [
    "run_id", "method", "sample_idx", "question_id", "image_id",
    "question", "predicted", "vqa_accuracy",
    "energy_gross_j", "energy_net_j", "avg_power_w",
    "idle_power_w", "elapsed_s", "tokens_generated",
    "n_visual_tokens", "ckpt1_dropped", "ckpt1_deferred",
    "ckpt2_dropped", "ckpt2_deferred", "ckpt3_dropped",
    "timestamp",
]
writer = csv.DictWriter(log_f, fieldnames=fieldnames)
if write_header:
    writer.writeheader()

for run_id in range(1, N_RUNS + 1):
    if run_id in manifest["completed_runs"]:
        print(f"Run {run_id} already complete -- skipping.")
        continue

    print(f"\n{'='*50}\nStarting run {run_id} of {N_RUNS}\n{'='*50}")
    manifest["current_run"] = run_id
    save_manifest(manifest)

    for s in SAMPLES:
        fname = s["image_file"]
        question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
        pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=MAX_TILES).to(torch.float16).cuda()

        for method, ckpt_layers in [("unpruned_matched", []), ("dap", CHECKPOINT_LAYERS)]:
            window_name = f"run{run_id}_{method}_{s['sample_idx']}"
            monitor.begin_window(window_name)
            t0 = time.perf_counter()

            predicted, n_tok, gen_log = dap_generate(
                model, tokenizer, pixel_values, question_text,
                max_new_tokens=MAX_NEW_TOKENS, checkpoint_layers=ckpt_layers
            )

            elapsed = time.perf_counter() - t0
            measurement = monitor.end_window(window_name)

            energy_gross = measurement.total_energy
            avg_power = energy_gross / elapsed if elapsed > 0 else 0
            energy_net = max((avg_power - IDLE_POWER_W) * elapsed, 0)
            acc = vqa_accuracy(predicted, s["answers"])

            ckpts = gen_log["checkpoints"]
            ckpt1_dropped = ckpts[0]["dropped"] if len(ckpts) > 0 else None
            ckpt1_deferred = ckpts[0]["deferred"] if len(ckpts) > 0 else None
            ckpt2_dropped = ckpts[1]["dropped"] if len(ckpts) > 1 else None
            ckpt2_deferred = ckpts[1]["deferred"] if len(ckpts) > 1 else None
            ckpt3_dropped = ckpts[2]["dropped"] if len(ckpts) > 2 else None

            writer.writerow({
                "run_id": run_id, "method": method, "sample_idx": s["sample_idx"],
                "question_id": s["question_id"], "image_id": s["image_id"],
                "question": s["question"], "predicted": predicted,
                "vqa_accuracy": round(acc, 4),
                "energy_gross_j": round(energy_gross, 4),
                "energy_net_j": round(energy_net, 4),
                "avg_power_w": round(avg_power, 3),
                "idle_power_w": round(IDLE_POWER_W, 3),
                "elapsed_s": round(elapsed, 4),
                "tokens_generated": n_tok,
                "n_visual_tokens": gen_log["n_visual_tokens"],
                "ckpt1_dropped": ckpt1_dropped, "ckpt1_deferred": ckpt1_deferred,
                "ckpt2_dropped": ckpt2_dropped, "ckpt2_deferred": ckpt2_deferred,
                "ckpt3_dropped": ckpt3_dropped,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })

        del pixel_values
        gc.collect()
        torch.cuda.empty_cache()

    manifest["completed_runs"].append(run_id)
    manifest["current_run"] = None
    save_manifest(manifest)
    print(f"Run {run_id} complete.")

log_f.close()
print("\nAll runs complete.")